# 00 — Canonical time-series profile

Run this notebook **after Notebook 01 or 02** has materialised
`SPEC-CORE`. It explores standardised telemetry, so the same code profiles
telecom and Petrobras 3W without remembering native column names.

The notebook is deliberately label-free:

- it opens `SPEC-CORE` only;
- it profiles the first 40% of the selected entities' histories;
- it never opens `SPEC-EVAL`;
- it records descriptive evidence, not detector thresholds.

Outputs are a compact metric profile, series-quality table, decomposition
evidence and figures. Notebook 03 reads the metric profile before scoring.

## 1. Setup

Set `PROFILE_SECTOR` to `telecom` or `petrobras_3w`. Expensive diagnostics
use a small deterministic entity sample; exact canonical row counts come from
the materialisation manifest.

In [ ]:
import hashlib
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_TABLES,
    CORE_VERSION,
    sha256_file,
    write_json,
)

SECTOR = os.getenv("PROFILE_SECTOR", "telecom")
assert SECTOR in {"telecom", "petrobras_3w"}

default_run_id = (
    "telecom_v4_1_full_v1"
    if SECTOR == "telecom"
    else "contract_challenge_v1"
)
CORE_RUN_ROOT = Path(os.getenv(
    "PROFILE_CORE_RUN_ROOT",
    str(
        DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
        / SECTOR / default_run_id
    ),
))
CORE = CORE_RUN_ROOT / "SPEC-CORE"
PROFILE_ROOT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "profiles" / SECTOR / CORE_RUN_ROOT.name
)
FIGURE_ROOT = PROFILE_ROOT / "figures"

PROFILE_ENTITY_LIMIT = int(os.getenv("PROFILE_ENTITY_LIMIT", "8"))
PROFILE_FRACTION = float(os.getenv("PROFILE_FRACTION", "0.40"))
RANDOM_SEED = 42

assert 0 < PROFILE_FRACTION < 1
if PROFILE_ROOT.exists():
    raise FileExistsError(
        f"Profile already exists: {PROFILE_ROOT}. "
        "Remove it deliberately or use a new core run."
    )
FIGURE_ROOT.mkdir(parents=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

print("SPEC-CORE:", CORE)
print("Profile output:", PROFILE_ROOT)

## 2. Load the standardised contract and select entities

In [ ]:
manifest = json.loads((CORE / "manifest.json").read_text())
catalogue = pd.read_parquet(CORE / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE / "entity_registry.parquet")
relations = pd.read_parquet(CORE / "entity_relations.parquet")
operational_events = pd.read_parquet(
    CORE / "operational_events.parquet"
)
collection_gaps = pd.read_parquet(
    CORE / "collection_gaps.parquet"
)

assert set(manifest["row_counts"]) == set(CORE_TABLES)
assert not any(
    str(column).startswith("gt_")
    for column in catalogue.columns
)
assert CORE.name == "SPEC-CORE"

leaf_type = "ont" if SECTOR == "telecom" else "oil_well"
leaf_entities = sorted(
    registry.loc[
        registry["entity_type"].eq(leaf_type),
        "entity_id",
    ].astype(str).unique()
)

def stable_key(entity_id):
    text = f"{RANDOM_SEED}|{entity_id}".encode()
    return hashlib.sha256(text).hexdigest()


selected_entities = sorted(
    leaf_entities,
    key=stable_key,
)[: min(PROFILE_ENTITY_LIMIT, len(leaf_entities))]
if not selected_entities:
    raise ValueError(f"No {leaf_type} entities found")

display(pd.Series({
    "contract_version": manifest["contract_version"],
    "sector": SECTOR,
    "canonical_rows": manifest["row_counts"]["telemetry"],
    "canonical_metrics": len(catalogue),
    "leaf_entities": len(leaf_entities),
    "profile_entities": len(selected_entities),
    "profile_fraction": PROFILE_FRACTION,
    "cadence_seconds": manifest.get(
        "cadence_seconds", "inferred after telemetry load"
    ),
}, name="value").to_frame())
display(catalogue)
print("Selected entities:", selected_entities)

In [ ]:
telemetry_columns = [
    "event_ts", "entity_id", "metric_id", "value",
    "quality_code", "exposure",
]
frames = []
for part in sorted((CORE / "telemetry").glob("part-*.parquet")):
    frame = pd.read_parquet(part, columns=telemetry_columns)
    frame = frame.loc[
        frame["entity_id"].astype(str).isin(selected_entities)
    ]
    if len(frame):
        frames.append(frame)

telemetry = pd.concat(frames, ignore_index=True)
telemetry["event_ts"] = pd.to_datetime(
    telemetry["event_ts"],
    utc=True,
)
telemetry["value"] = pd.to_numeric(
    telemetry["value"],
    errors="coerce",
)
telemetry["exposure"] = pd.to_numeric(
    telemetry["exposure"],
    errors="coerce",
)

cadence_seconds = manifest.get("cadence_seconds")
if cadence_seconds is None:
    time_differences = (
        telemetry.sort_values(["entity_id", "event_ts"])
        .groupby("entity_id")["event_ts"]
        .diff()
        .dropna()
        .dt.total_seconds()
    )
    time_differences = time_differences.loc[
        time_differences.gt(0)
    ]
    if time_differences.empty:
        raise ValueError("Could not infer a positive canonical cadence")
    cadence_seconds = float(time_differences.mode().iloc[0])
cadence = pd.Timedelta(seconds=float(cadence_seconds))

cutoffs = {}
for entity_id, group in telemetry.groupby("entity_id"):
    times = group["event_ts"].drop_duplicates().sort_values()
    position = max(0, int(len(times) * PROFILE_FRACTION) - 1)
    cutoffs[str(entity_id)] = times.iloc[position]

cutoff_by_row = telemetry["entity_id"].astype(str).map(cutoffs)
profile_mask = telemetry["event_ts"].le(cutoff_by_row)
profile_frame = telemetry.loc[profile_mask].copy()
profile_frame = profile_frame.merge(
    catalogue[[
        "metric_id", "measurement_kind",
        "expected_behaviour_profile", "candidate_periods",
        "anomaly_direction",
    ]],
    on="metric_id",
    how="left",
    validate="many_to_one",
)

assert profile_frame["measurement_kind"].notna().all()
assert not any(
    column.startswith("gt_") for column in profile_frame.columns
)

display(profile_frame.head())
print(
    f"Profile frame: {len(profile_frame):,} rows, "
    f"{profile_frame.entity_id.nunique()} entities, "
    f"{profile_frame.metric_id.nunique()} metrics"
)

## 3. Canonical structure and quality

`quality_code=invalid` means a row was observed but its value was null.
`collection_gaps` describes expected timestamps that were absent. These are
different conditions and are reported separately.

In [ ]:
display(
    profile_frame.groupby(
        ["metric_id", "quality_code"],
        dropna=False,
    ).size().rename("rows").to_frame()
)

selected_gaps = collection_gaps.loc[
    collection_gaps["entity_id"].astype(str).isin(selected_entities)
].copy()
display(selected_gaps.head(20))

selected_events = operational_events.loc[
    operational_events["entity_id"].astype(str).isin(
        selected_entities
    )
].copy()
display(
    selected_events["event_type"]
    .value_counts()
    .rename("events")
    .to_frame()
)

In [ ]:
def longest_constant_run(values):
    values = pd.to_numeric(values, errors="coerce")
    new_run = (
        values.ne(values.shift())
        | values.isna()
        | values.shift().isna()
    )
    lengths = values.notna().groupby(new_run.cumsum()).sum()
    return int(lengths.max()) if len(lengths) else 0


quality_rows = []
for (entity_id, metric_id), group in profile_frame.groupby(
    ["entity_id", "metric_id"],
    sort=True,
):
    group = group.sort_values("event_ts")
    timestamps = group["event_ts"].drop_duplicates()
    duration = timestamps.max() - timestamps.min()
    expected = int(duration / cadence) + 1
    values = group["value"]
    quality_rows.append({
        "entity_id": str(entity_id),
        "metric_id": metric_id,
        "rows": len(group),
        "expected_rows": expected,
        "grid_coverage": timestamps.nunique() / expected,
        "invalid_fraction": group["quality_code"].eq("invalid").mean(),
        "clipped_fraction": group["quality_code"].eq("clipped").mean(),
        "unique_observed_values": values.nunique(dropna=True),
        "longest_constant_run": longest_constant_run(values),
        "degenerate": values.dropna().nunique() <= 1,
    })

series_quality = pd.DataFrame(quality_rows)
display(
    series_quality.groupby("metric_id")[[
        "grid_coverage", "invalid_fraction", "clipped_fraction",
        "longest_constant_run",
    ]].describe(percentiles=[0.05, 0.50, 0.95])
)

In [ ]:
quality_matrix = series_quality.pivot(
    index="metric_id",
    columns="entity_id",
    values="invalid_fraction",
)
figure, axis = plt.subplots(
    figsize=(max(8, len(selected_entities)), 7)
)
sns.heatmap(
    quality_matrix,
    cmap="magma_r",
    vmin=0,
    vmax=max(0.01, float(quality_matrix.max().max())),
    ax=axis,
)
axis.set_title("Invalid-value fraction by metric and entity")
figure.tight_layout()
figure.savefig(
    FIGURE_ROOT / "01_invalid_value_heatmap.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

## 4. Robust metric distributions

In [ ]:
def robust_mad(values):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return np.nan
    return float(
        1.4826 * np.median(np.abs(values - values.median()))
    )


metric_rows = []
for metric_id, group in profile_frame.groupby("metric_id", sort=True):
    values = group["value"].dropna()
    entity_medians = group.groupby("entity_id")["value"].median()
    within_mad = group.groupby("entity_id")["value"].apply(robust_mad)
    kind = group["measurement_kind"].iloc[0]
    has_exposure = group["exposure"].gt(0).any()

    if kind == "interval_count":
        recommended_transform = (
            "log_exposure_rate" if has_exposure else "log1p_count"
        )
    elif kind == "cumulative_counter":
        recommended_transform = "log1p_increment"
    elif kind == "bounded_fraction":
        recommended_transform = "log10_floor"
    else:
        recommended_transform = "identity"

    metric_rows.append({
        "metric_id": metric_id,
        "measurement_kind": kind,
        "recommended_transform": recommended_transform,
        "rows": len(group),
        "entities": group["entity_id"].nunique(),
        "invalid_fraction": group["quality_code"].eq("invalid").mean(),
        "clipped_fraction": group["quality_code"].eq("clipped").mean(),
        "minimum": values.min(),
        "p01": values.quantile(0.01),
        "median": values.median(),
        "p99": values.quantile(0.99),
        "maximum": values.max(),
        "mad": robust_mad(values),
        "skewness": values.skew(),
        "excess_kurtosis": values.kurt(),
        "zero_fraction": values.eq(0).mean(),
        "between_entity_mad": robust_mad(entity_medians),
        "median_within_entity_mad": within_mad.median(),
        "between_within_ratio": (
            robust_mad(entity_medians) / within_mad.median()
            if within_mad.median() > 0
            else np.nan
        ),
    })

metric_profile = pd.DataFrame(metric_rows)
display(metric_profile)

In [ ]:
focus_metrics = (
    metric_profile.sort_values(
        ["measurement_kind", "invalid_fraction", "metric_id"],
        ascending=[True, False, True],
    )
    .groupby("measurement_kind", as_index=False)
    .head(2)["metric_id"]
    .head(6)
    .tolist()
)

figure, axes = plt.subplots(
    len(focus_metrics),
    2,
    figsize=(14, 3.2 * len(focus_metrics)),
    squeeze=False,
)
for row, metric_id in enumerate(focus_metrics):
    values = profile_frame.loc[
        profile_frame["metric_id"].eq(metric_id),
        "value",
    ].dropna()
    if len(values) > 50_000:
        values = values.sample(50_000, random_state=RANDOM_SEED)
    low, high = values.quantile([0.01, 0.99])
    sns.histplot(
        values.clip(low, high),
        kde=True,
        ax=axes[row, 0],
    )
    axes[row, 0].set_title(
        f"{metric_id}: 1st–99th percentile view"
    )
    sns.boxplot(x=values, ax=axes[row, 1])
    axes[row, 1].axvline(
        values.quantile(0.95),
        color="darkorange",
        linestyle="--",
    )
    axes[row, 1].set_title(f"{metric_id}: full-scale boxplot")

figure.tight_layout()
figure.savefig(
    FIGURE_ROOT / "02_metric_distributions.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

## 5. Complete series, rolling statistics and dependence

Rolling windows stay inside contiguous timestamp segments. They never bridge
a collection gap.

In [ ]:
representative_entity = selected_entities[0]
representative = (
    profile_frame.loc[
        profile_frame["entity_id"].astype(str).eq(
            representative_entity
        )
        & profile_frame["metric_id"].isin(focus_metrics)
    ]
    .pivot(index="event_ts", columns="metric_id", values="value")
    .sort_index()
)

figure, axes = plt.subplots(
    len(focus_metrics),
    1,
    figsize=(15, 2.7 * len(focus_metrics)),
    sharex=True,
    squeeze=False,
)
for row, metric_id in enumerate(focus_metrics):
    series = representative[metric_id]
    segment = series.index.to_series().diff().gt(cadence * 1.5).cumsum()
    one_day_window = max(
        5, int(pd.Timedelta("1 day") / cadence)
    )
    rolling_window = min(
        one_day_window, max(5, len(series) // 10)
    )
    rolling = series.groupby(segment).rolling(
        rolling_window,
        min_periods=5,
    )
    rolling_median = rolling.median().reset_index(level=0, drop=True)
    rolling_iqr = (
        rolling.quantile(0.75).reset_index(level=0, drop=True)
        - rolling.quantile(0.25).reset_index(level=0, drop=True)
    )
    axes[row, 0].plot(
        series.index,
        series,
        linewidth=0.6,
        alpha=0.6,
        label="observed",
    )
    axes[row, 0].plot(
        rolling_median.index,
        rolling_median,
        label="rolling median",
    )
    axes[row, 0].plot(
        rolling_iqr.index,
        rolling_iqr,
        label="rolling IQR",
        linestyle="--",
    )
    axes[row, 0].set_ylabel(metric_id)
    axes[row, 0].legend(loc="upper right")

axes[0, 0].set_title(
    f"Canonical series and rolling statistics: {representative_entity}"
)
figure.tight_layout()
figure.savefig(
    FIGURE_ROOT / "03_rolling_statistics.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

In [ ]:
level_correlation = representative.corr(method="spearman")
change_correlation = representative.diff().corr(method="spearman")

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(
    level_correlation,
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
    square=True,
    ax=axes[0],
)
axes[0].set_title("Spearman correlation: levels")
sns.heatmap(
    change_correlation,
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
    square=True,
    ax=axes[1],
)
axes[1].set_title("Spearman correlation: first differences")
figure.tight_layout()
figure.savefig(
    FIGURE_ROOT / "04_feature_dependence.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

## 6. STL and residual autocorrelation

Candidate periods come from the sector pack. A decomposition is fitted only
when a complete regular segment contains at least five cycles. A `log1p`
scale is used only for a non-negative series whose rolling standard deviation
rises with its rolling level.

In [ ]:
PERIOD_SECONDS = {
    "P1D": 86_400,
    "P7D": 604_800,
    "P365D": 31_536_000,
}

def longest_regular_segment(group):
    group = (
        group[["event_ts", "value"]]
        .sort_values("event_ts")
        .drop_duplicates("event_ts")
        .reset_index(drop=True)
    )
    new_segment = (
        group["value"].isna()
        | group["value"].shift().isna()
        | group["event_ts"].diff().ne(cadence)
    )
    group["segment"] = new_segment.cumsum()
    observed = group.dropna(subset=["value"])
    if observed.empty:
        return pd.Series(dtype=float)
    largest = observed.groupby("segment").size().idxmax()
    segment = observed.loc[observed["segment"].eq(largest)]
    return segment.set_index("event_ts")["value"].astype(float)


def candidate_period_samples(raw_periods, series_length):
    if raw_periods is None or (
        isinstance(raw_periods, float) and np.isnan(raw_periods)
    ):
        return None
    if isinstance(raw_periods, str):
        raw_periods = [raw_periods]
    for name in list(raw_periods):
        seconds = PERIOD_SECONDS.get(str(name))
        if seconds is None:
            continue
        period = int(round(seconds / cadence.total_seconds()))
        if period >= 2 and series_length >= 5 * period:
            return period
    return None

In [ ]:
decomposition_rows = []
diagnostic_objects = []

for metric_id, metric_group in profile_frame.groupby(
    "metric_id",
    sort=True,
):
    raw_periods = metric_group["candidate_periods"].iloc[0]
    for entity_id, entity_group in metric_group.groupby(
        "entity_id",
        sort=True,
    ):
        series = longest_regular_segment(entity_group)
        period = candidate_period_samples(raw_periods, len(series))
        if period is None or series.std() == 0:
            continue

        rolling_level = series.rolling(period).mean()
        rolling_std = series.rolling(period).std()
        level_scale_correlation = rolling_level.corr(rolling_std)
        use_log = (
            series.min() >= 0
            and pd.notna(level_scale_correlation)
            and level_scale_correlation > 0.30
        )
        model_series = np.log1p(series) if use_log else series
        result = STL(
            model_series,
            period=period,
            robust=True,
        ).fit()

        residual_variance = np.var(result.resid, ddof=1)
        seasonal_denominator = np.var(
            result.seasonal + result.resid,
            ddof=1,
        )
        trend_denominator = np.var(
            result.trend + result.resid,
            ddof=1,
        )
        decomposition_rows.append({
            "entity_id": str(entity_id),
            "metric_id": metric_id,
            "observations": len(series),
            "period_samples": period,
            "scale": "log1p" if use_log else "identity",
            "level_scale_correlation": level_scale_correlation,
            "seasonal_strength": (
                max(0, 1 - residual_variance / seasonal_denominator)
                if seasonal_denominator > 0
                else np.nan
            ),
            "trend_strength": (
                max(0, 1 - residual_variance / trend_denominator)
                if trend_denominator > 0
                else np.nan
            ),
            "residual_mad": (
                1.4826 * np.median(
                    np.abs(result.resid - np.median(result.resid))
                )
            ),
            "residual_acf1": pd.Series(result.resid).autocorr(1),
        })
        diagnostic_objects.append({
            "entity_id": str(entity_id),
            "metric_id": metric_id,
            "series": model_series,
            "result": result,
            "period": period,
        })

decomposition_columns = [
    "entity_id", "metric_id", "observations", "period_samples",
    "scale", "level_scale_correlation", "residual_mad",
    "residual_acf1", "seasonal_strength", "trend_strength",
]
decomposition = pd.DataFrame(
    decomposition_rows,
    columns=decomposition_columns,
)
display(decomposition)

In [ ]:
plotted_metrics = set()
for item in diagnostic_objects:
    if (
        item["metric_id"] not in focus_metrics
        or item["metric_id"] in plotted_metrics
    ):
        continue
    plotted_metrics.add(item["metric_id"])

    figure = item["result"].plot()
    figure.set_size_inches(12, 8)
    figure.suptitle(
        f"{item['metric_id']} — robust STL — "
        f"{item['entity_id']}",
        y=1.02,
    )
    figure.tight_layout()
    safe_name = item["metric_id"].replace(".", "_")
    figure.savefig(
        FIGURE_ROOT / f"05_stl_{safe_name}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

    maximum_lag = min(100, len(item["series"]) // 4)
    figure, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(
        item["series"],
        lags=maximum_lag,
        ax=axes[0],
    )
    plot_pacf(
        item["result"].resid,
        lags=maximum_lag,
        method="ywm",
        ax=axes[1],
    )
    axes[0].set_title("Observed ACF")
    axes[1].set_title("STL residual PACF")
    figure.tight_layout()
    figure.savefig(
        FIGURE_ROOT / f"06_acf_pacf_{safe_name}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

## 7. Final metric profile

In [ ]:
if not decomposition.empty:
    decomposition_summary = (
        decomposition.groupby("metric_id", as_index=False)
        .agg(
            decomposed_entities=("entity_id", "nunique"),
            median_period_samples=("period_samples", "median"),
            median_seasonal_strength=("seasonal_strength", "median"),
            median_trend_strength=("trend_strength", "median"),
            median_residual_mad=("residual_mad", "median"),
            median_residual_acf1=("residual_acf1", "median"),
        )
    )
    metric_profile = metric_profile.merge(
        decomposition_summary,
        on="metric_id",
        how="left",
        validate="one_to_one",
    )
else:
    decomposition_summary = pd.DataFrame()

quality_summary = (
    series_quality.groupby("metric_id", as_index=False)
    .agg(
        median_grid_coverage=("grid_coverage", "median"),
        worst_grid_coverage=("grid_coverage", "min"),
        maximum_constant_run=("longest_constant_run", "max"),
        degenerate_series=("degenerate", "sum"),
    )
)
metric_profile = metric_profile.merge(
    quality_summary,
    on="metric_id",
    how="left",
    validate="one_to_one",
)

display(metric_profile)

In [ ]:
metric_profile.to_parquet(
    PROFILE_ROOT / "metric_profile.parquet",
    index=False,
)
series_quality.to_parquet(
    PROFILE_ROOT / "series_quality.parquet",
    index=False,
)
decomposition.to_parquet(
    PROFILE_ROOT / "decomposition_evidence.parquet",
    index=False,
)
pd.DataFrame([
    {"entity_id": entity_id, "profile_end": cutoff}
    for entity_id, cutoff in cutoffs.items()
]).to_parquet(
    PROFILE_ROOT / "profile_windows.parquet",
    index=False,
)

profile_manifest = {
    "profile_version": "canonical_profile_v1",
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "spec_core": str(CORE),
    "spec_core_manifest_sha256": sha256_file(
        CORE / "manifest.json"
    ),
    "metric_catalogue_sha256": sha256_file(
        CORE / "metric_catalogue.parquet"
    ),
    "spec_eval_opened": False,
    "profile_fraction": PROFILE_FRACTION,
    "cadence_seconds": float(cadence_seconds),
    "selected_entities": selected_entities,
    "metric_count": len(metric_profile),
    "profile_rows": len(profile_frame),
    "recommended_transforms_are_kind_based": True,
    "seasonality_is_evidence_not_an_automatic_model_action": True,
    "figures": sorted(
        path.name for path in FIGURE_ROOT.glob("*.png")
    ),
}
write_json(
    PROFILE_ROOT / "profile_manifest.json",
    profile_manifest,
)

print("Saved canonical profile:", PROFILE_ROOT)

## What this profile does not claim

- It does not estimate anomaly prevalence.
- It does not choose an alert threshold.
- It does not call STL residuals “noise”; they may contain anomalies.
- It does not force seasonality onto short 3W event instances.
- It does not automatically remove a seasonal component in Notebook 03.
- It does not use topology to infer causes.

Those are modelling or evaluation decisions, not profiling tasks.